# 02_02 TF-IDF: why the book and scikit-learn disagree

Counts treat "is" and "spooky" as equally informative. **TF-IDF** weights each count by how rare the word
is across the documents. The chapter worked it by hand; this notebook does the same arithmetic in code,
finds that scikit-learn gets different numbers, and pins down the three reasons exactly. The same weights,
refined, are what BM25 ranks by in the next notebook.

**How this notebook works.** The same rhythm as Lab 01:

1. **Recall.** Answer from memory before you look anything up. `ask()` tells you at once whether you were right.
2. **Predict, then run.** Before a cell with a surprise in it, write your prediction into `guess()`. The next cell runs the code and `reveal()` compares.
3. **Worked example, then your turn.** One example is done in full; the next, near-identical one has lines marked `# YOUR CODE HERE`.
4. **Check.** A `check_...()` cell tests what you saved, exactly as the checkpoint will, and says what to fix.

Run cells in order with **Shift+Enter**. If you get lost, **Kernel, Restart Kernel and Run All Cells** starts clean.

Running this in Google Colab? This cell sets it up; in CourseLabs it does nothing.

In [ ]:
# Colab setup. In a CourseLabs session this cell does nothing.
import os, sys
if "google.colab" in sys.modules:
    import importlib, importlib.util, subprocess
    LAB, REPO = "lab-nlp-02-turning-words-into-numbers", "/content/nlp-course"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/fenago/nlp-course.git", REPO], check=True)
    os.chdir(f"{REPO}/{LAB}")
    if not os.path.exists("data"):
        os.symlink("../data", "data")
    os.makedirs("out", exist_ok=True)
    os.environ["NLPLAB_DATA"] = f"{REPO}/data"
    sys.path.insert(0, os.getcwd())
    PIP = {'bm25s': 'bm25s',
           'sentence_transformers': 'sentence-transformers',
           'sklearn': 'scikit-learn',
           'pandas': 'pandas',
           'numpy': 'numpy'}
    missing = [spec for mod, spec in PIP.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
        importlib.invalidate_caches()
    print(f"Ready: {LAB} and its data are in {os.getcwd()}; installed {len(missing)} package(s).")
elif not os.path.isdir("/opt/nlplab/data") and os.path.isdir("data"):
    # A downloaded copy on your own computer: the helpers read data/ from here.
    os.environ["NLPLAB_DATA"] = os.path.abspath("data")

In [ ]:
import csv
import json
import math
import os
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nlpcheck import ask, guess, reveal, check_02_02

reviews3 = ["This movie is very scary and long",
            "This movie is not scary and is slow",
            "This movie is spooky and good"]
kw = list(csv.DictReader(open("data/kittiwake_reviews.csv")))
print(len(kw), "Kittiwake app reviews;", kw[0]["stars"], "stars:", kw[0]["text"])
os.makedirs("out", exist_ok=True)

## 1. Recall

**r3.** How many words are in the vocabulary of the three horror reviews? (a number)

**r4.** Why are "dog bites man" and "man bites dog" the same bag of words?
(a) they have the same length, (b) CountVectorizer sorts words, (c) a bag keeps counts and drops order

In [ ]:
ask("r3", "")
ask("r4", "")

## 2. The book's numbers

The chapter's definitions, applied to review 2, "This movie is not scary and is slow":

- **term frequency**: `tf = count of the word in the review / number of words in the review`,
- **inverse document frequency**: `idf = log10(number of reviews / number of reviews containing the word)`,
- **TF-IDF** is their product.

In [ ]:
docs = [r.lower().split() for r in reviews3]
N = len(docs)
r2 = docs[1]
print(f"{'word':8} {'tf':>6} {'df':>3} {'idf':>6} {'tf-idf':>7}")
for w in dict.fromkeys(r2):
    tf = r2.count(w) / len(r2)
    df = sum(w in d for d in docs)
    idf = math.log10(N / df)
    print(f"{w:8} {tf:6.3f} {df:3} {idf:6.3f} {tf * idf:7.3f}")

`this`, `movie`, `is` and `and` are in every review, so their IDF is `log10(3/3) = 0` and they vanish.
`not` and `slow` appear only here, and score highest. `scary` is shared with review 1 and scores in
between.

## 3. scikit-learn's numbers

Predict: what TF-IDF will scikit-learn's `TfidfVectorizer` give `this` in review 2? The book says 0.

In [ ]:
guess("sklearn_this_r2", None)   # a number

In [ ]:
tv = TfidfVectorizer()
T = tv.fit_transform(reviews3)
names = tv.get_feature_names_out()
table = pd.DataFrame(T.toarray(), columns=names, index=["review 1", "review 2", "review 3"]).round(3)
display(table)
reveal("sklearn_this_r2", round(float(table.loc["review 2", "this"]), 3))
print("idf:", dict(zip(names, tv.idf_.round(3))))

`0.264`, not zero. Three differences, all deliberate:

1. **Natural log, plus one.** scikit-learn's IDF is `ln(n / df) + 1`. The `+ 1` means a word in every
   document keeps an IDF of 1 and a small weight, rather than being deleted. A search for "this movie"
   should still find something.
2. **Smoothing.** By default (`smooth_idf=True`) it pretends there is one extra document containing every
   word once: `ln((1 + n) / (1 + df)) + 1`. That keeps the division safe for a word no document contains.
3. **Raw counts, then length one.** Its TF is the raw count, not the count divided by the review's length.
   Instead, at the end, each row is divided by its own length (the **L2 norm**, the square root of the sum
   of squares), so every review becomes a vector of length 1. That is what makes long and short
   documents comparable in the next notebook.

Turn smoothing and normalisation off and the IDFs become plain `ln(n / df) + 1`:

In [ ]:
raw = TfidfVectorizer(smooth_idf=False, norm=None).fit(reviews3)
print(dict(zip(raw.get_feature_names_out(), raw.idf_.round(3))))
print("ln(3/1) + 1 =", round(math.log(3) + 1, 3), "| ln(3/2) + 1 =", round(math.log(1.5) + 1, 3))

## 4. Worked example: review 1 by hand, scikit-learn's way

In [ ]:
def sk_idf(word):
    df = sum(word in d for d in docs)
    return math.log((1 + N) / (1 + df)) + 1

r1 = docs[0]
weights = {w: r1.count(w) * sk_idf(w) for w in dict.fromkeys(r1)}   # raw count times IDF
length = math.sqrt(sum(v * v for v in weights.values()))            # the L2 norm
by_hand = {w: v / length for w, v in weights.items()}
for w, v in by_hand.items():
    print(f"{w:6} by hand {v:.3f}   scikit-learn {table.loc['review 1', w]:.3f}")

Every value matches. Nothing in `TfidfVectorizer` is a black box any more.

## 5. Your turn: review 2

Do the same for review 2, which has the extra wrinkle that `is` appears twice. Build `tfidf_r2`, a dict
from each distinct word of review 2 to its scikit-learn TF-IDF, computed by hand with `sk_idf`.

In [ ]:
r2 = docs[1]
tfidf_r2 = {}   # YOUR CODE HERE: weights, the L2 length, then divide

for w, v in tfidf_r2.items():
    print(f"{w:6} by hand {v:.3f}   scikit-learn {table.loc['review 2', w]:.3f}")
json.dump(tfidf_r2, open("out/02_02_tfidf.json", "w"), indent=1)
check_02_02();

Look at `is` in review 2: it is in every review, so its IDF is 1, but its count is 2, so it ends up with
a larger weight (`0.527`) than `scary`. TF-IDF has not made "is" meaningful; it has only turned it down,
and a repeated common word can still outweigh a rarer one. Remember that when the next notebook searches.

## 6. Exit ticket

**x2.** When is a word's TF-IDF high? (a) frequent in this document and rare in the others,
(b) frequent everywhere, (c) rare everywhere

In [ ]:
ask("x2", "")

Explain it back: why does scikit-learn add 1 to the IDF, and what would a search engine lose without it?

*Your explanation:* 